In [86]:

import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from datetime import datetime

In [87]:
# PARTE 1: Carga y Exploración de Datos

def cargar_datos() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Carga los datos de estudiantes, calificaciones y materias.
    """
    # Datos de estudiantes
    estudiantes = pd.DataFrame({
        'boleta': ['2021630001', '2021630002', '2021630003', '2021630004', '2021630005',
                   '2022630001', '2022630002', '2022630003', '2022630004', '2022630005',
                   '2023630001', '2023630002', '2023630003', '2023630004', '2023630005'],
        'nombre': ['Juan Pérez García', 'María López Ruiz', 'Pedro Sánchez Torres',
                   'Ana Martínez Díaz', 'Luis Rodríguez Vega', 'Carmen Flores Luna',
                   'Roberto Díaz Mora', 'Laura Torres Silva', 'Diego Ramírez Cruz',
                   'Sofía Vargas Romo', 'Carlos Mendoza Ríos', 'Patricia Ortiz León',
                   'Miguel Ángel Castro', 'Fernanda Reyes Paz', 'Andrés Guzmán Villa'],
        'semestre': [4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2],
        'carrera': ['CD'] * 15,
        'email': ['juan.perez@ipn.mx', 'maria.lopez@ipn.mx', 'pedro.sanchez@ipn.mx',
                  'ana.martinez@ipn.mx', 'luis.rodriguez@ipn.mx', 'carmen.flores@ipn.mx',
                  'roberto.diaz@ipn.mx', 'laura.torres@ipn.mx', 'diego.ramirez@ipn.mx',
                  'sofia.vargas@ipn.mx', 'carlos.mendoza@ipn.mx', 'patricia.ortiz@ipn.mx',
                  'miguel.castro@ipn.mx', 'fernanda.reyes@ipn.mx', 'andres.guzman@ipn.mx']
    })
    
    # Datos de materias
    materias = pd.DataFrame({
        'materia_id': ['MAT101', 'MAT102', 'PROG101', 'PROG102', 'EST101', 'EST102', 'BD101'],
        'nombre': ['Cálculo Diferencial', 'Cálculo Integral', 'Programación I',
                   'Programación II', 'Probabilidad', 'Estadística Inferencial',
                   'Bases de Datos'],
        'creditos': [8, 8, 6, 6, 6, 6, 6],
        'semestre_materia': [1, 2, 1, 2, 2, 3, 3]
    })
    # Generar calificaciones (simuladas)
    np.random.seed(42)
    calificaciones_data = []
    
    for boleta in estudiantes['boleta']:
        semestre = estudiantes[estudiantes['boleta'] == boleta]['semestre'].values[0]
        materias_cursadas = materias[materias['semestre_materia'] <= semestre]['materia_id'].tolist()
        
        for materia in materias_cursadas:
            # Generar calificaciones aleatorias (con algunos casos especiales)
            base = np.random.uniform(5, 10)
            p1 = round(min(10, max(0, base + np.random.normal(0, 1))), 1)
            p2 = round(min(10, max(0, base + np.random.normal(0, 1))), 1)
            final = round(min(10, max(0, base + np.random.normal(0, 0.5))), 1)
            
            # Algunos valores nulos aleatorios
            if np.random.random() < 0.05:
                p2 = np.nan
            
            calificaciones_data.append({
                'boleta': boleta,
                'materia_id': materia,
                'parcial_1': p1,
                'parcial_2': p2,
                'final': final
            })
    
    calificaciones = pd.DataFrame(calificaciones_data)
    
    return estudiantes, calificaciones, materias

In [88]:
def info_general(df_estudiantes: pd.DataFrame, df_calificaciones: pd.DataFrame) -> Dict:
    """
    Genera información general del sistema.
    """
    resultado = {
        "total_estudiantes": len(df_estudiantes),
        "total_registros_calif": len(df_calificaciones),
        "semestres": sorted(df_estudiantes["semestre"].unique().tolist()),
        "materias_con_registros": df_calificaciones["materia_id"].nunique()
    }
    
    # TODO: Implementar
    
    return resultado

In [89]:
def validar_datos(df_calificaciones: pd.DataFrame) -> Dict:
    columnas = ["parcial_1", "parcial_2", "final"]

    nulos = df_calificaciones[columnas].isna().any(axis=1)

    fuera_rango = (
        (df_calificaciones[columnas] < 0) |
        (df_calificaciones[columnas] > 10)
    ).any(axis=1)

    resultado = {
        "registros_con_nulos": nulos.sum(),
        "calificaciones_fuera_rango": fuera_rango.sum(),
        "datos_validos": not (nulos.any() or fuera_rango.any())
    }
    """
    # TODO: Implementar
    # Tip: Usa .isna().any(axis=1) para detectar filas con nulos
    # Tip: Para fuera de rango, revisa parcial_1, parcial_2, final
    """
    return resultado

In [90]:
def buscar_estudiante(df_estudiantes: pd.DataFrame, criterio: str, valor: str) -> pd.DataFrame:
    """
    Busca estudiantes por diferentes criterios.
    """

    if criterio == "nombre":
        return df_estudiantes[
            df_estudiantes["nombre"].str.contains(valor, case=False, na=False)
        ]

    elif criterio == "boleta":
        return df_estudiantes[
            df_estudiantes["boleta"] == valor
        ]

    elif criterio == "semestre":
        return df_estudiantes[
            df_estudiantes["semestre"] == int(valor)
        ]

    return pd.DataFrame()

In [91]:
def obtener_kardex(boleta: str, df_estudiantes: pd.DataFrame,
                   df_calificaciones: pd.DataFrame, df_materias: pd.DataFrame) -> Dict:
    """
    Obtiene el kardex completo de un estudiante.
    """

    resultado = {
        "estudiante": None,
        "materias": None,
        "promedio_general": None,
        "creditos_cursados": None,
        "materias_aprobadas": None,
        "materias_reprobadas": None
    }

    # 1. Datos del estudiante
    estudiante = df_estudiantes[df_estudiantes["boleta"] == boleta]

    if estudiante.empty:
        return resultado

    resultado["estudiante"] = estudiante.iloc[0].to_dict()

    # 2. Calificaciones del estudiante
    califs = df_calificaciones[df_calificaciones["boleta"] == boleta].copy()

    # 3. Unir con materias
    materias = califs.merge(df_materias, on="materia_id", how="left")

    # 4. Promedio por materia
    materias["promedio"] = (
        materias["parcial_1"] +
        materias["parcial_2"] +
        materias["final"]
    ) / 3

    resultado["materias"] = materias

    # Promedio general
    resultado["promedio_general"] = materias["promedio"].mean()

    # 5. Aprobadas y reprobadas
    resultado["materias_aprobadas"] = (materias["promedio"] >= 6).sum()
    resultado["materias_reprobadas"] = (materias["promedio"] < 6).sum()

    # Créditos cursados
    if "creditos" in materias.columns:
        resultado["creditos_cursados"] = materias["creditos"].sum()

    return resultado

In [92]:
def filtrar_por_rendimiento(df_calificaciones: pd.DataFrame,
                            df_estudiantes: pd.DataFrame,
                            min_promedio: float = None,
                            max_promedio: float = None) -> pd.DataFrame:
    """
    Filtra estudiantes por rango de promedio.
    """

    # Promedio por materia
    df = df_calificaciones.copy()

    df["promedio"] = (
        df["parcial_1"] +
        df["parcial_2"] +
        df["final"]
    ) / 3

    # Promedio general por estudiante
    promedios = (
        df.groupby("boleta")["promedio"]
        .mean()
        .reset_index()
        .rename(columns={"promedio": "promedio_general"})
    )

    # Aplicar filtros
    if min_promedio is not None:
        promedios = promedios[
            promedios["promedio_general"] >= min_promedio
        ]

    if max_promedio is not None:
        promedios = promedios[
            promedios["promedio_general"] <= max_promedio
        ]

    # Unir con estudiantes
    resultado = promedios.merge(
        df_estudiantes,
        on="boleta",
        how="left"
    )

    return resultado

In [93]:
def calcular_promedio_materia(df_calificaciones: pd.DataFrame, materia_id: str) -> Dict:
    """
    Calcula estadísticas de una materia.
    """

    resultado = {
        "materia": materia_id,
        "inscritos": None,
        "promedio_parcial1": None,
        "promedio_parcial2": None,
        "promedio_final": None,
        "promedio_general": None,
        "tasa_aprobacion": None,
        "calificacion_maxima": None,
        "calificacion_minima": None
    }

    materia = df_calificaciones[
        df_calificaciones["materia_id"] == materia_id
    ].copy()

    if materia.empty:
        return resultado

    materia["promedio"] = (
        materia["parcial_1"] +
        materia["parcial_2"] +
        materia["final"]
    ) / 3

    resultado["inscritos"] = len(materia)
    resultado["promedio_parcial1"] = materia["parcial_1"].mean()
    resultado["promedio_parcial2"] = materia["parcial_2"].mean()
    resultado["promedio_final"] = materia["final"].mean()
    resultado["promedio_general"] = materia["promedio"].mean()

    resultado["tasa_aprobacion"] = (
        (materia["promedio"] >= 6).sum() / len(materia)
    ) * 100

    resultado["calificacion_maxima"] = materia["promedio"].max()
    resultado["calificacion_minima"] = materia["promedio"].min()

    return resultado

In [94]:
def ranking_estudiantes(df_calificaciones: pd.DataFrame,
                        df_estudiantes: pd.DataFrame,
                        top_n: int = 10) -> pd.DataFrame:
    """
    Genera ranking de mejores estudiantes por promedio.
    """

    df = df_calificaciones.copy()

    # Promedio por materia
    df["promedio"] = (
        df["parcial_1"] +
        df["parcial_2"] +
        df["final"]
    ) / 3

    # Promedio general por estudiante
    ranking = (
        df.groupby("boleta")["promedio"]
        .mean()
        .reset_index()
        .rename(columns={"promedio": "promedio_general"})
    )

    # Ordenar de mayor a menor
    ranking = ranking.sort_values(
        by="promedio_general",
        ascending=False
    )

    # Top N
    ranking = ranking.head(top_n)

    # Agregar datos del estudiante
    ranking = ranking.merge(
        df_estudiantes,
        on="boleta",
        how="left"
    )

    return ranking

In [95]:
def estadisticas_por_semestre(df_estudiantes: pd.DataFrame,
                              df_calificaciones: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula estadísticas agrupadas por semestre.
    """

    df = df_calificaciones.copy()

    # Promedio por materia
    df["promedio"] = (
        df["parcial_1"] +
        df["parcial_2"] +
        df["final"]
    ) / 3

    # Promedio general por estudiante
    promedios = (
        df.groupby("boleta")["promedio"]
        .mean()
        .reset_index()
        .rename(columns={"promedio": "promedio_general"})
    )

    # Unir con semestre
    promedios = promedios.merge(
        df_estudiantes[["boleta", "semestre"]],
        on="boleta",
        how="left"
    )

    # Estadísticas por semestre
    estadisticas = (
        promedios.groupby("semestre")["promedio_general"]
        .agg([
            "count",
            "mean",
            "std",
            "min",
            "max"
        ])
        .reset_index()
    )

    estadisticas.columns = [
        "semestre",
        "num_estudiantes",
        "promedio_semestre",
        "desviacion_std",
        "promedio_minimo",
        "promedio_maximo"
    ]

    return estadisticas

In [96]:
def identificar_estudiantes_riesgo(df_calificaciones: pd.DataFrame,
                                   df_estudiantes: pd.DataFrame,
                                   umbral_promedio: float = 7.0,
                                   max_reprobadas: int = 2) -> pd.DataFrame:
    """
    Identifica estudiantes en riesgo académico.
    """

    df = df_calificaciones.copy()

    # Promedio por materia
    df["promedio_materia"] = (
        df["parcial_1"] +
        df["parcial_2"] +
        df["final"]
    ) / 3

    # Promedio general por estudiante
    promedio_general = (
        df.groupby("boleta")["promedio_materia"]
        .mean()
        .reset_index()
        .rename(columns={"promedio_materia": "promedio_general"})
    )

    # Materias reprobadas por estudiante
    reprobadas = (
        df[df["promedio_materia"] < 6]
        .groupby("boleta")
        .size()
        .reset_index(name="materias_reprobadas")
    )

    # Unir información
    riesgo = promedio_general.merge(
        reprobadas,
        on="boleta",
        how="left"
    )

    riesgo["materias_reprobadas"] = riesgo["materias_reprobadas"].fillna(0)

    # Identificar motivo
    def obtener_motivo(fila):
        bajo_promedio = fila["promedio_general"] < umbral_promedio
        muchas_reprobadas = fila["materias_reprobadas"] > max_reprobadas

        if bajo_promedio and muchas_reprobadas:
            return "Ambos"
        elif bajo_promedio:
            return "Bajo promedio"
        elif muchas_reprobadas:
            return "Materias reprobadas"
        else:
            return None

    riesgo["motivo"] = riesgo.apply(obtener_motivo, axis=1)

    # Solo estudiantes en riesgo
    riesgo = riesgo[riesgo["motivo"].notna()]

    # Agregar datos del estudiante
    riesgo = riesgo.merge(
        df_estudiantes,
        on="boleta",
        how="left"
    )

    return riesgo

In [97]:
def generar_reporte_academico(df_estudiantes: pd.DataFrame,
                              df_calificaciones: pd.DataFrame,
                              df_materias: pd.DataFrame) -> Dict:
    """
    Genera reporte académico completo.
    """

    reporte = {
        "resumen_general": {},
        "por_semestre": None,
        "por_materia": None,
        "mejores_estudiantes": None,
        "estudiantes_riesgo": None,
        "fecha_generacion": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

    # Resumen general
    reporte["resumen_general"] = {
        "total_estudiantes": len(df_estudiantes),
        "total_materias": len(df_materias),
        "total_calificaciones": len(df_calificaciones)
    }

    # Estadísticas por semestre
    reporte["por_semestre"] = estadisticas_por_semestre(
        df_estudiantes,
        df_calificaciones
    )

    # Estadísticas por materia
    estadisticas_materias = []

    for materia_id in df_materias["materia_id"]:
        estadisticas_materias.append(
            calcular_promedio_materia(
                df_calificaciones,
                materia_id
            )
        )

    reporte["por_materia"] = pd.DataFrame(estadisticas_materias)

    # Ranking de estudiantes
    reporte["mejores_estudiantes"] = ranking_estudiantes(
        df_calificaciones,
        df_estudiantes,
        top_n=10
    )

    # Estudiantes en riesgo
    reporte["estudiantes_riesgo"] = identificar_estudiantes_riesgo(
        df_calificaciones,
        df_estudiantes
    )

    return reporte

In [98]:
def exportar_kardex(boleta: str, kardex: Dict, formato: str = 'csv') -> str:
    """
    Exporta el kardex de un estudiante a archivo.
    """

    fecha = datetime.now().strftime("%Y%m%d_%H%M%S")

    if formato.lower() == "csv":

        nombre_archivo = f"kardex_{boleta}_{fecha}.csv"

        if kardex["materias"] is not None:
            kardex["materias"].to_csv(
                nombre_archivo,
                index=False
            )

    elif formato.lower() == "json":

        nombre_archivo = f"kardex_{boleta}_{fecha}.json"

        datos = {
            "estudiante": kardex["estudiante"],
            "promedio_general": kardex["promedio_general"],
            "creditos_cursados": kardex["creditos_cursados"],
            "materias_aprobadas": kardex["materias_aprobadas"],
            "materias_reprobadas": kardex["materias_reprobadas"]
        }

        with open(nombre_archivo, "w", encoding="utf-8") as archivo:
            json.dump(
                datos,
                archivo,
                ensure_ascii=False,
                indent=4,
                default=str
            )

    else:
        raise ValueError("Formato no soportado. Use 'csv' o 'json'")

    return nombre_archivo

In [99]:
def mostrar_kardex(kardex: Dict) -> None:
    """Muestra el kardex de forma legible."""
    if kardex['estudiante'] is None:
        print("❌ Estudiante no encontrado")
        return
    
    est = kardex['estudiante']
    print("=" * 70)
    print("                         KARDEX ACADÉMICO")
    print("=" * 70)
    print(f"\n📋 DATOS DEL ESTUDIANTE")
    print("-" * 40)
    print(f"Boleta: {est.get('boleta', 'N/A')}")
    print(f"Nombre: {est.get('nombre', 'N/A')}")
    print(f"Semestre: {est.get('semestre', 'N/A')}")
    print(f"Carrera: {est.get('carrera', 'N/A')}")
    print(f"Email: {est.get('email', 'N/A')}")
    
    print(f"\n📚 CALIFICACIONES")
    print("-" * 70)
    if kardex['materias'] is not None and len(kardex['materias']) > 0:
        print(kardex['materias'].to_string(index=False))
    else:
        print("Sin calificaciones registradas")
    
    print(f"\n📊 RESUMEN")
    print("-" * 40)
    print(f"Promedio General: {kardex.get('promedio_general', 0):.2f}")
    print(f"Créditos Cursados: {kardex.get('creditos_cursados', 0)}")
    print(f"Materias Aprobadas: {kardex.get('materias_aprobadas', 0)}")
    print(f"Materias Reprobadas: {kardex.get('materias_reprobadas', 0)}")
    print("=" * 70)

In [100]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte académico completo."""
    print("=" * 70)
    print("              REPORTE ACADÉMICO - CIENCIA DE DATOS")
    print(f"              Generado: {reporte['fecha_generacion']}")
    print("=" * 70)
    
    # Resumen general
    res = reporte.get('resumen_general', {})
    print(f"\n📊 RESUMEN GENERAL")
    print("-" * 40)
    print(f"Total de estudiantes: {res.get('total_estudiantes', 'N/A')}")
    print(f"Promedio global: {res.get('promedio_global', 0):.2f}")
    print(f"Tasa de aprobación: {res.get('tasa_aprobacion', 0):.1f}%")
    
    # Por semestre
    if reporte.get('por_semestre') is not None:
        print(f"\n📅 ESTADÍSTICAS POR SEMESTRE")
        print("-" * 40)
        print(reporte['por_semestre'].to_string())
    
    # Mejores estudiantes
    if reporte.get('mejores_estudiantes') is not None:
        print(f"\n🏆 TOP 5 ESTUDIANTES")
        print("-" * 40)
        print(reporte['mejores_estudiantes'].head().to_string(index=False))
    
    # Estudiantes en riesgo
    if reporte.get('estudiantes_riesgo') is not None and len(reporte['estudiantes_riesgo']) > 0:
        print(f"\n⚠️ ESTUDIANTES EN RIESGO ({len(reporte['estudiantes_riesgo'])})")
        print("-" * 40)
        print(reporte['estudiantes_riesgo'].to_string(index=False))
    else:
        print(f"\n✅ No hay estudiantes en riesgo académico")
    
    print("\n" + "=" * 70)

In [101]:
# Cargar datos
df_estudiantes, df_calificaciones, df_materias = cargar_datos()

print("DATOS CARGADOS")
print("=" * 50)
print(f"\nEstudiantes ({len(df_estudiantes)} registros):")
print(df_estudiantes.head())

print(f"\nCalificaciones ({len(df_calificaciones)} registros):")
print(df_calificaciones.head())

print(f"\nMaterias ({len(df_materias)} registros):")
print(df_materias)

DATOS CARGADOS

Estudiantes (15 registros):
       boleta                nombre  semestre carrera                  email
0  2021630001     Juan Pérez García         4      CD      juan.perez@ipn.mx
1  2021630002      María López Ruiz         4      CD     maria.lopez@ipn.mx
2  2021630003  Pedro Sánchez Torres         4      CD   pedro.sanchez@ipn.mx
3  2021630004     Ana Martínez Díaz         4      CD    ana.martinez@ipn.mx
4  2021630005   Luis Rodríguez Vega         4      CD  luis.rodriguez@ipn.mx

Calificaciones (95 registros):
       boleta materia_id  parcial_1  parcial_2  final
0  2021630001     MAT101        5.8        7.2    7.0
1  2021630001     MAT102        6.1        4.5    4.8
2  2021630001    PROG101        3.9        7.5    6.9
3  2021630001    PROG102        4.9        5.8    5.4
4  2021630001     EST101        8.6        6.4    6.1

Materias (7 registros):
  materia_id                   nombre  creditos  semestre_materia
0     MAT101      Cálculo Diferencial         8

In [102]:
# Prueba de información general
print("\nINFORMACIÓN GENERAL")
print("=" * 50)
info = info_general(df_estudiantes, df_calificaciones)
print(info)


INFORMACIÓN GENERAL
{'total_estudiantes': 15, 'total_registros_calif': 95, 'semestres': [2, 3, 4], 'materias_con_registros': 7}


In [103]:
# Prueba de validación
print("\nVALIDACIÓN DE DATOS")
print("=" * 50)
validacion = validar_datos(df_calificaciones)
print(validacion)


VALIDACIÓN DE DATOS
{'registros_con_nulos': np.int64(5), 'calificaciones_fuera_rango': np.int64(0), 'datos_validos': False}


In [104]:
# Prueba de búsqueda
print("\nBÚSQUEDA DE ESTUDIANTES")
print("=" * 50)

print("\n-- Buscar por nombre 'María' --")
resultado = buscar_estudiante(df_estudiantes, 'nombre', 'María')
print(resultado)

print("\n-- Buscar por semestre 3 --")
resultado = buscar_estudiante(df_estudiantes, 'semestre', '3')
print(resultado)


BÚSQUEDA DE ESTUDIANTES

-- Buscar por nombre 'María' --
       boleta            nombre  semestre carrera               email
1  2021630002  María López Ruiz         4      CD  maria.lopez@ipn.mx

-- Buscar por semestre 3 --
       boleta              nombre  semestre carrera                 email
5  2022630001  Carmen Flores Luna         3      CD  carmen.flores@ipn.mx
6  2022630002   Roberto Díaz Mora         3      CD   roberto.diaz@ipn.mx
7  2022630003  Laura Torres Silva         3      CD   laura.torres@ipn.mx
8  2022630004  Diego Ramírez Cruz         3      CD  diego.ramirez@ipn.mx
9  2022630005   Sofía Vargas Romo         3      CD   sofia.vargas@ipn.mx


In [105]:
# Prueba de kardex
print("\nKARDEX DE ESTUDIANTE")
print("=" * 50)
kardex = obtener_kardex('2021630001', df_estudiantes, df_calificaciones, df_materias)
mostrar_kardex(kardex)


KARDEX DE ESTUDIANTE
                         KARDEX ACADÉMICO

📋 DATOS DEL ESTUDIANTE
----------------------------------------
Boleta: 2021630001
Nombre: Juan Pérez García
Semestre: 4
Carrera: CD
Email: juan.perez@ipn.mx

📚 CALIFICACIONES
----------------------------------------------------------------------
    boleta materia_id  parcial_1  parcial_2  final                  nombre  creditos  semestre_materia  promedio
2021630001     MAT101        5.8        7.2    7.0     Cálculo Diferencial         8                 1  6.666667
2021630001     MAT102        6.1        4.5    4.8        Cálculo Integral         8                 2  5.133333
2021630001    PROG101        3.9        7.5    6.9          Programación I         6                 1  6.100000
2021630001    PROG102        4.9        5.8    5.4         Programación II         6                 2  5.366667
2021630001     EST101        8.6        6.4    6.1            Probabilidad         6                 2  7.033333
2021630001

In [106]:
# Prueba de ranking
print("\nRANKING DE ESTUDIANTES")
print("=" * 50)
ranking = ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=5)
print(ranking)
     


RANKING DE ESTUDIANTES
       boleta  promedio_general               nombre  semestre carrera  \
0  2023630003          8.906667  Miguel Ángel Castro         2      CD   
1  2023630004          8.593333   Fernanda Reyes Paz         2      CD   
2  2021630005          8.438095  Luis Rodríguez Vega         4      CD   
3  2021630004          8.386667    Ana Martínez Díaz         4      CD   
4  2022630003          8.255556   Laura Torres Silva         3      CD   

                   email  
0   miguel.castro@ipn.mx  
1  fernanda.reyes@ipn.mx  
2  luis.rodriguez@ipn.mx  
3    ana.martinez@ipn.mx  
4    laura.torres@ipn.mx  


In [107]:
# Prueba de reporte completo
print("\nREPORTE ACADÉMICO COMPLETO")
reporte = generar_reporte_academico(df_estudiantes, df_calificaciones, df_materias)
mostrar_reporte(reporte)


REPORTE ACADÉMICO COMPLETO
              REPORTE ACADÉMICO - CIENCIA DE DATOS
              Generado: 2026-06-23 21:22:35

📊 RESUMEN GENERAL
----------------------------------------
Total de estudiantes: 15
Promedio global: 0.00
Tasa de aprobación: 0.0%

📅 ESTADÍSTICAS POR SEMESTRE
----------------------------------------
   semestre  num_estudiantes  promedio_semestre  desviacion_std  promedio_minimo  promedio_maximo
0         2                5           7.823111        0.975832         6.460000         8.906667
1         3                5           7.807302        0.307451         7.409524         8.255556
2         4                5           7.398286        0.984443         6.195238         8.438095

🏆 TOP 5 ESTUDIANTES
----------------------------------------
    boleta  promedio_general              nombre  semestre carrera                 email
2023630003          8.906667 Miguel Ángel Castro         2      CD  miguel.castro@ipn.mx
2023630004          8.593333  Fernanda Reye